# Level 2 — Vectorization, Error Analysis, and Numerical Reliability

## Scientific Objectives
1. Compare standard Python loops against NumPy vectorization for ET computation.
2. Demonstrate floating-point artifacts.
3. Model the propagation of sensor error (noise) through our irrigation logic.

In [ ]:
import numpy as np
import pandas as pd
import time
import matplotlib.pyplot as plt
from pathlib import Path

# Load data
weather = pd.read_csv(Path('.').resolve().parent / 'data' / 'processed' / 'weather_cleaned.csv')

# Extract to native lists and numpy arrays
temps = weather['temperature_c'].tolist()
winds = weather['wind_speed_mps'].tolist()
solars = weather['solar_index'].tolist()

temps_np = np.array(temps)
winds_np = np.array(winds)
solars_np = np.array(solars)

In [ ]:
# 1. VECTORIZATION VS LOOPS PERFORMANCE

def et_loop(t_list, w_list, s_list):
    results = []
    for i in range(len(t_list)):
        et = 0.12 * t_list[i] + 0.35 * w_list[i] + 2.4 * s_list[i]
        results.append(max(0, et))
    return results

def et_vectorized(t_arr, w_arr, s_arr):
    et = 0.12 * t_arr + 0.35 * w_arr + 2.4 * s_arr
    return np.maximum(0, et)

# Scaling up data to simulate 100,000 farm zones to show real time differences
scale = 100000
large_temps = temps * scale
large_winds = winds * scale
large_solars = solars * scale
large_temps_np = np.array(large_temps)
large_winds_np = np.array(large_winds)
large_solars_np = np.array(large_solars)

start = time.time()
res_loop = et_loop(large_temps, large_winds, large_solars)
loop_time = time.time() - start

start = time.time()
res_vec = et_vectorized(large_temps_np, large_winds_np, large_solars_np)
vec_time = time.time() - start

print("=== PERFORMANCE COMPARISON ===")
print(f"Standard Python Loop Time: {loop_time:.4f} seconds")
print(f"NumPy Vectorized Time:     {vec_time:.4f} seconds")
print(f"Speedup Factor:            {loop_time / vec_time:.1f}x faster")

In [ ]:
# 2. FLOATING POINT ERROR DEMONSTRATION
print("\n=== FLOATING POINT BEHAVIOR ===")
print("In base-2 binary, fractions like 0.1 and 0.2 have repeating expansions.")
print(f"Is 0.1 + 0.2 equal to 0.3? -> {0.1 + 0.2 == 0.3}")
print(f"Actual value of 0.1 + 0.2  -> {0.1 + 0.2:.17f}")
print("Resolution: Always use np.isclose() or math.isclose() for scientific equality checks.")
print(f"Using np.isclose: -> {np.isclose(0.1 + 0.2, 0.3)}")

In [ ]:
# 3. ERROR PROPAGATION IN IRRIGATION SENSORS
# We will model how a +/- 3% random measurement noise on the soil sensor 
# propagates into an over-irrigation or under-irrigation error over a month.

true_soil_moisture = np.linspace(35, 18, 30) # Linearly drying soil
wilting_point = 22.0

# Add normal Gaussian noise to simulate cheap sensor error
np.random.seed(42)
sensor_noise = np.random.normal(0, 1.5, 30)
noisy_soil_moisture = true_soil_moisture + sensor_noise

plt.figure(figsize=(10, 5))
plt.plot(true_soil_moisture, label="True Soil Moisture", color='green', linewidth=2)
plt.plot(noisy_soil_moisture, label="Noisy Sensor Reading", color='red', alpha=0.6, marker='x')
plt.axhline(wilting_point, color='black', linestyle='--', label="Irrigation Trigger (22%)")
plt.title("Error Propagation: Sensor Noise Impacting Irrigation Timing")
plt.xlabel("Days")
plt.ylabel("Soil Moisture (%)")
plt.legend()
plt.show()

true_trigger_day = np.argmax(true_soil_moisture < wilting_point)
noisy_trigger_day = np.argmax(noisy_soil_moisture < wilting_point)
print(f"Without noise, irrigation triggers on Day {true_trigger_day}.")
print(f"With sensor noise, irrigation triggers prematurely on Day {noisy_trigger_day}.")
print(f"Conclusion: Measurement error propagates directly into wasted pump energy and water.")